## `Agenda:`

#### `[#1]` **Find out why the ATaCR correction plot outputs look like the corrections were never applied or are not working.**

#### `[#2]` **Explicitly verify the functionality and correctness of the transfer function in Py-ATaCR**

#### `[#3]` **Get back to finishing the notebook for the complete ATaCR method with all the intermediate analysis (tfs, coh, etc.).**
____
____

____
# `[#1]` **Find out why the ATaCR correction plot outputs look like the corrections were never applied or are not working.**

____
Summary: This was a pretty quick mystery to solve.

It appears to have been largely bad plotting for a particular station-event with the majority of the apparent changes after correction outside (<150s) the band that was plotted (150-0.5) and thus the TFs were applied as intended all along. Whether those TFs calculations were written correctly or not is what I will look at in the next section.
____


First, a basic description of the problem: We have seen on some events what appeared to be plots showing a complete lack of event correction.

Specifically, an M07A event we looked at last month:

<img src="./znb_images/812evcorr.png" width="800">

##### So I'll begin with the simplest theory and see if we need to go any further:

That it's just not plotting very well or its bad events and everything up to that point is fine. | 
---------|
A quick test for this is comparing with the ATaCR demo outputs in the github documentation.


Below, plots on the left are from my build and ones on the right are from the events demo in the ATaCR documentation at https://nfsi-canada.github.io/OBStools/atacr.html#tutorial.

_____

Some things to keep in mind when looking at these:

1. This is a different station-event pair (is not M07A). 

2. I have no way to directly confirm the exact days used in the ATaCR demo figure images. That said, the documentation states the spectra used is the entire month of March 2012, so I will use the same days.

3. The original ATaCR build doesn't use a taper at the TF stage, mine does.

![demo_compare_staavg.png](./znb_images/demo_compare_staavg.png)

![demo_compare_dayavg.png](./znb_images/demo_compare_dayavg.png)

##### `Note`: I know looking at the example above motivates questions (again) on how the addition of TF tapers to Python ATaCR is changing the results. I won't dive into that in this notebook but maybe we can talk about it? I don't see anything too suprising for the above in that regard, just kind of interesting.

### Nonetheless, let's atleast compare the TFs:

Aside from the taper, the two plots above and below should be almost indistinguishable from eachother. And they are until you look extremely close. The only thing I can think of that would create even the smallest differences between these two plots is the fact that I have moved the the remove_response out of the download step (and to the start of the next ATaCR step) so that I can save the raw data to SAC files. The consequence is instrument response is removed from a 32-bit trace (SAC) instead of a 64-bit trace (miniseed). I could keep everything as a miniseed but there's no chance i'd have a large enough hard drive to handle that scale.

Should we care about this? I don't think we should.

##### `Question`: What exactly is the spectral slope shown in these transfer functions actually telling us?

![TransferFunctionsComp_M08A.png](./znb_images/TransferFunctionsComp_M08A.png)

____
In total, while all of this is admittedly anecdotal it's still a strong indication that, aside from the TF taper and what day-correction TF is loaded, ATaCR is essentially identical to what it is in the original build.

With that in mind, issues we were seeing with the other station-event pairs looking like a correction was never applied may have been some combination of bad plotting or bad events.

___
#### So, here is some better plotting:

Returning to the M07A event again, I've widened the bandpass out to 250s (from 150s) in the hopes that atleast the tilt noise corrections would show a little more clearly.

##### `Note`: I noticed a very low amplitude ringing after I widened to 250s so I applied a 5% taper prior to filtering here which seemed to stop it. I know this was a Gibb's effect and not a longer period component of the trace because it was accompanied with a large amplitude wavelet confined to both ends of the trace.

##### `Note`: Yes, both the pre- (gray) and post- (black) corrected traces get the exact same filter.

![demo_compare_dayavg.png](./znb_images/250sM07Acompw150s.png)

### There's the (right) elusive tilt/comp corrections we couln't see in the original (left) M07A plot. 

### From a time domain assertion, the relative lack of compliance noise this would imply ATaCR's is reducing at shorter than 150s for M07A (1356m depth) is somewhat concerning. 
### Some pre-/post- PSDs will shed some light on this.

## PRE- POST- VERTICAL PSDs

___
# `[#2]` **Verify the functionality and correctness of the transfer functions in Py-ATaCR**

#### `What we discussed on last month's Notebook:`
So as I brought up last month, I'm a bit concerned whether or not the TFs are applying the equations correctly and despite the plotting explanation I still need to look into it. I've done the analytic summary of the proof I am suspicious is being applied incorrectly last month and I will break it down more in depth here. 

However, the priority right now is to begin with first finding another indication besides the equations just looking wonky that this is actually happening.

#### `Describing what I'm worried might be happening based on what I saw in the TF code and the easiest ways I can confirm it: `

This section is to suggest an alternative to a meticulous QA on the TF code unless I really have to, which is to look at the post-correction traces and spectra. We basically talked about this being the next logical step already but I wanted to lay out the arguments for it here with more detail.

At the simplest level, the product of a correct compliance noise TF multiplied by a pressure trace in pressure units (/Hz) should output vertical spectra in velocity units (/Hz) that contains all coherent noise shown in both to be subtracted from an event spectra. That is the basic mechanics of a compliance correction. For a ZP-12 (and ZP-H) TF, the same is still true with the TF product with a pressure trace now producing noise spectra (in units of vertical velocity) that is coherent across both the vertical, horizontal, and pressure components spectra.

If any TF is built incorrectly, the post-correction trace amplitudes will not make sense. In the simplest example possible, a TF subtracting pressure noise from seismic as opposed to subtracting seismic noise (from pressure).

Nonetheless, this transfer function area of the code is dense with complex naming structure for variable names that contain ZERO code comments on the majority of it (e.g. gc2cP_c1, which is 2P CSD with the effect of 1 spectra removed i believe).

Getting around this code problem, I can just look at the output corrected events and spectra. If the ampltides and spectra make sense, then we are probably (?) okay.

___

`Note:` **_There is notable apparent inconsistencies between the code and the paper's they are built on (e.g. Crawford&Webb) but those may be accounted for earlier in the code and I wouldn't know for sure without digging deep into it because, again, nothing is commented._** 

#### `Question:` Should we be applying the square of the TF during event deconvolution? Both the ML-ATaCR and Py-ATaCR codes do this and I'm not sure why based on Crawford & Webb / Bell, 2015 who don't appear to do that.


#### `What I want to do about it right now: Synthetic Traces`

As suggested, approaching this instead from the results end is alot simpler which, again, I think is where we left it just before I took a break for comps.

We didn't talk about it then but I'm hoping just comparing the outputs with synthetic traces might be a quick answer to define what is an acceptable output for this issue I'm concerned with.

Synthetic traces from teleseismic events come with a list of assumptions (model structure, isotropy, etc.) that will scale their usefulness here, but all I am looking for is if the amplitudes after correction are just completely off from what they should be. Comparing with a synthetic isn't a perfect solution but they are quick/easy if we just want an idea of what amplitudes we should be seeing across any given event.

___

A few things to keep in mind when looking at these:
1. These are synthetic traces (red) for an MW 6.6 teleseismic event (https://ds.iris.edu/spud/momenttensor/734501) that occured on March 09, 2012 that was observed on both stations (M07A,M08A) in this notebook.

2. The model used is the 3600s anisotropic PREM at 2-100s resolution (prem_a_2s, https://ds.iris.edu/ds/products/syngine/#earth). The synthetic traces are generated with InstaSeis/AxiSEM (https://github.com/krischer/instaseis) via IRIS's Syngine breqfast shell (https://service.iris.edu/irisws/syngine/1/).

3. Following the run time and resolution of the model, all traces are filtered (2-100s) then trimmed (3600s).

4. I really wish I knew a way to build some synthetic pressure data as this could of gotten alot more interesting (e.g. synthetic TFs using an aprox. of tilt).

M08A_20120690709.Synthetic

![M08A_20120690709.Synthetic](./znb_images/M08A_20120690709.Synthetic_StaAvg.png)


M07A_20120690709.Synthetic

![M07A_20120690709.Synthetic_StaAvg](./znb_images/M07A_20120690709.Synthetic_StaAvg.png)


A few things stand out here:
1. Comparing the Z1 (first row) and ZP (last row) corrections, the loss of the surface wave coda after the ZP correction would definitely suggest that specifically the compliance correction is removing event structure. This wasn't a goal of concern for this notebook but its probably something we should look into again (tapers?).

2. Aside from this idea of the correction removing event structure, the amplitudes overall look pretty reasonable. Specifically the first ariving P around 750s is within a few percent of the synthetic amplitude. So, atleast nothing in the amplitudes is blowing up (down?) which I would suspect if the TFs were written incorrectly. That said, until that issue of the TF removing event structure is understood better this approach to confirming the TFs derivation being correct or not won't completely work.

#### In closing, I'm not totally convinced from this that the TFs are written correctly, but I am atleast less worried after seeing these anecdotal results. If we want to keep pursuing this thread, the next step is to focus on the issue of possible retention of event structure after correction.

___
# `[#3]` **Get back to finishing the notebook for the complete ATaCR method with all it's intermediate analyses (tfs, coh, etc.).**

____